# TP3 - Exploración de datos

In [1]:
!pip -q install huggingface_hub scikit-learn

import pandas as pd
from huggingface_hub import hf_hub_download

ruta = hf_hub_download(
    repo_id="SINAI/ALIA-es-discriminative-stance-detection",
    filename="corpus_stance_detection_ALIA.jsonl",
    repo_type="dataset"
)

df = pd.read_json(ruta, lines=True)
print(df.shape)
display(df.head())

corpus_stance_detection_ALIA.jsonl:   0%|          | 0.00/4.48M [00:00<?, ?B/s]

(3000, 8)


,id,target,description,comment,annotation_1,annotation_2,annotation_3,majority_label
0,1,"""Bajar"" las motos de las aceras.",Esta propuesta es originaria del vecino. [NOMB...,A mi no me molesta tanto que aparquen en la ac...,neutral,against,neutral,neutral
1,2,"""Bajar"" las motos de las aceras.",Esta propuesta es originaria del vecino. [NOMB...,Existe una ordenanza municipal que autoriza y ...,against,against,against,against
2,3,"""Bajar"" las motos de las aceras.","Los motoristas, aparte del ruido a veces intol...",Las motos son la solución para descongestionar...,against,neutral,favor,
3,4,"""Bajar"" las motos de las aceras.","Los motoristas, aparte del ruido a veces intol...",Aquí todo se soluciona con multas e impuestos....,neutral,against,against,against
4,5,ARREGLAR LOS REBAJES DE LOS BORDILLOS PARA FAC...,El mal estado de las calles y aceras ha hecho ...,La accesibilidad no es un concepto de patente ...,favor,favor,favor,favor


In [2]:
# Distribución de etiquetas
print(df["majority_label"].value_counts(dropna=False))

print("\nPorcentaje por clase (sin los casos sin mayoría):")
print((df["majority_label"].value_counts(normalize=True) * 100).round(1))

# Longitud de los comentarios
longitudes = df["comment"].fillna("").str.len()
print("\nLongitud mediana:", longitudes.median())
print("Longitud promedio:", round(longitudes.mean(), 1))

# Acuerdo entre los tres anotadores
cols = ["annotation_1", "annotation_2", "annotation_3"]
n_distintas = df[cols].nunique(axis=1)
acuerdo_total = (n_distintas == 1).sum()
acuerdo_mayoria = (n_distintas == 2).sum()
desacuerdo_total = (n_distintas == 3).sum()

print("\nAcuerdo total:", acuerdo_total)
print("Acuerdo mayoritario:", acuerdo_mayoria)
print("Desacuerdo total:", desacuerdo_total)
print("Acuerdo unánime (%):", round(acuerdo_total / len(df) * 100, 1))

majority_label
favor      1780
against     627
neutral     443
            150
Name: count, dtype: int64

Porcentaje por clase (sin los casos sin mayoría):
majority_label
favor      59.3
against    20.9
neutral    14.8
            5.0
Name: proportion, dtype: float64

Longitud mediana: 266.0
Longitud promedio: 312.6

Acuerdo total: 1539
Acuerdo mayoritario: 1311
Desacuerdo total: 150
Acuerdo unánime (%): 51.3


In [3]:
from sklearn.model_selection import train_test_split

# Se eliminan los casos sin mayoría
preparado = df.dropna(subset=["majority_label"]).copy()

# Texto de entrada para todos los modelos
preparado["text"] = (
    "Propuesta: " + preparado["target"].fillna("") +
    "\nComentario: " + preparado["comment"].fillna("")
)

train, temp = train_test_split(
    preparado,
    test_size=0.30,
    random_state=42,
    stratify=preparado["majority_label"]
)

val, test = train_test_split(
    temp,
    test_size=0.50,
    random_state=42,
    stratify=temp["majority_label"]
)

print("Train:", len(train))
print("Validation:", len(val))
print("Test:", len(test))

Train: 2100
Validation: 450
Test: 450


In [4]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")
carpeta = Path("/content/drive/MyDrive/TP IA y Neuro/data")
carpeta.mkdir(parents=True, exist_ok=True)

# No sobrescribir archivos que ya usan los demás notebooks
for nombre, datos in [("train", train), ("val", val), ("test", test)]:
    archivo = carpeta / f"{nombre}.csv"
    if archivo.exists():
        print(nombre, "ya existe; no se modificó")
    else:
        datos.to_csv(archivo, index=False)
        print(nombre, "guardado")

Mounted at /content/drive
train ya existe; no se modificó
val ya existe; no se modificó
test ya existe; no se modificó
